# Stage 1: Data Pipeline
This notebook prepares and cleans the Nascenia AI Hackathon Bengali Medical Dialogue datasets, merges external data, and creates training/validation splits.

### 1. Install and Import Libraries

In [ ]:
# Install required packages if not already present
%pip install -q datasets pandas scikit-learn

import os
import pandas as pd
import sys

# Add src directory to path
sys.path.append(os.path.abspath('src'))

import data_utils
print("Modules imported successfully!")

### 2. Load and Profile Official Competition Data

In [ ]:
# Locate train.csv and test.csv
train_path = data_utils.find_data_file("train.csv")
test_path = data_utils.find_data_file("test.csv")

# Load dataframes
train_df, test_df = data_utils.load_official(train_path, test_path)

# Run profile analysis
data_utils.profile_official(train_df)

### 3. Clean Official Competition Data

In [ ]:
# Clean official train dataframe by removing boilerplate stubs and length outliers
cleaned_official_df = data_utils.clean_official(train_df)

print("Before cleaning:", len(train_df))
print("After cleaning:", len(cleaned_official_df))

### 4. Load and Normalize External Hugging Face Bengali Medical QA Dataset
This loads `shetumohanto/doctor_qa_bangla`, discovers its schema dynamically, parses/normalizes it into a standard format.

In [ ]:
# Load, inspect and normalize the external HF dataset
external_df = data_utils.load_external_normalized()
print("External dataset preview:")
print(external_df.head(3))

### 5. Merge and Deduplicate Official and External Datasets

In [ ]:
# Merge official cleaned data and external data while preventing prompts leakage
merged_df = data_utils.merge_and_dedupe(cleaned_official_df, external_df)

# Save the raw merged dataset (unfiltered version of merged data for references)
merged_df.to_csv("/kaggle/working/train_plus_external.csv", index=False)
print("Saved merged raw dataset to /kaggle/working/train_plus_external.csv")

### 6. Create SFT Train/Val Splits (Official Data Only)
Create splits on cleaned official data, ensuring validation matches the target domain (1000 validation rows stratified by output length).

In [ ]:
# Make splits
sft_train_df, sft_val_df = data_utils.make_splits(cleaned_official_df, val_size=1000, seed=42)

# Save train/val datasets
sft_train_df.to_csv("/kaggle/working/sft_train.csv", index=False)
sft_val_df.to_csv("/kaggle/working/sft_val.csv", index=False)
print("Saved training and validation splits to /kaggle/working/")

### 7. Prepare Cleaned Merged Dataset for Stage 1 Broad-Mix Training

In [ ]:
# Filter the merged dataset using the same word count constraints and quality checks
stage1_train_df = data_utils.clean_official(merged_df)

# Save Stage 1 training file
stage1_train_df.to_csv("/kaggle/working/train_plus_external_clean.csv", index=False)
print("Saved Stage 1 broad-mix training file to /kaggle/working/train_plus_external_clean.csv")

### 8. Final Outputs Summary Table

In [ ]:
import os

output_files = [
    "/kaggle/working/train_plus_external.csv",
    "/kaggle/working/sft_train.csv",
    "/kaggle/working/sft_val.csv",
    "/kaggle/working/train_plus_external_clean.csv"
]

summary_data = []
for filepath in output_files:
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        df_temp = pd.read_csv(filepath)
        summary_data.append({
            "File Path": filepath,
            "Row Count": len(df_temp),
            "File Size (MB)": f"{size_mb:.2f} MB"
        })

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))